In [ ]:
import os
import json
import requests
import time
import re
from google.colab import drive

# 1. Conexión con Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Rutas de los archivos en tu Drive
path_json_maestro = '/content/drive/MyDrive/TFE_Maestría/datos_idearq.json'
path_destino_imagenes = '/content/drive/MyDrive/TFE_Maestría/Dataset_CARL/Originales'

os.makedirs(path_destino_imagenes, exist_ok=True)

# 2. Cargar el archivo JSON automáticamente desde el Drive
try:
    with open(path_json_maestro, 'r', encoding='utf-8') as f:
        yacimientos = json.load(f)
    print(f"Archivo JSON cargado con éxito. Se procesarán {len(yacimientos)} bloques de yacimientos.")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo en la ruta: {path_json_maestro}")
    print("Asegúrate de subir el archivo 'datos_idearq.json' exactamente en esa carpeta de Drive.")
    yacimientos = []

def limpiar_nombre(texto):
    # Normaliza el texto para evitar errores con eñes, acentos o caracteres prohibidos en carpetas
    texto = re.sub(r'[áàäâ]', 'a', texto)
    texto = re.sub(r'[éèëê]', 'e', texto)
    texto = re.sub(r'[íìïî]', 'i', texto)
    texto = re.sub(r'[óòöô]', 'o', texto)
    texto = re.sub(r'[úùüû]', 'u', texto)
    texto = re.sub(r'[ñ]', 'n', texto)
    texto = re.sub(r'[^a-zA-Z0-9_\-\(\s)]', '', texto)
    return texto.strip().replace(' ', '_')

# 3. Bucle inteligente de descarga y ordenamiento
if yacimientos:
    print("Iniciando descarga masiva estructurada...")

    for bloque in yacimientos:
        # Extraer el nombre del panel o abrigo
        panel_nombre = limpiar_nombre(bloque.get('panel', f"Yaci_{bloque.get('id_yaci', 'sin_id')}"))

        # Separar las URLs (por comas) y descripciones (por almohadillas)
        raw_urls = bloque.get('urls', '').split(',')
        raw_desc = bloque.get('descripciones', '').split('#')

        if not bloque.get('urls'):
            continue  # Si el bloque no tiene URLs, saltar

        # Crear subcarpeta física para este yacimiento/panel en tu Drive
        ruta_carpeta_panel = os.path.join(path_destino_imagenes, panel_nombre)
        os.makedirs(ruta_carpeta_panel, exist_ok=True)

        print(f"\n Procesando: {panel_nombre} ({len(raw_urls)} imágenes)")

        for idx, url_completa in enumerate(raw_urls):
            url_completa = url_completa.strip()
            if not url_completa:
                continue

            # Obtener el ID numérico de la imagen (ej: 200623)
            id_img = url_completa.split('/')[-1].replace('h.jpg', '')

            # Emparejar con la descripción de la figura correspondiente de manera segura
            if idx < len(raw_desc):
                # Nos quedamos con la primera figura listada antes del punto y coma
                desc_limpia = raw_desc[idx].split(';')[0]
                meta_nombre = limpiar_nombre(desc_limpia)
            else:
                meta_nombre = "Figura_Indeterminada"

            # Nombre final y limpio: Panel_Figura_ID.jpg
            nombre_archivo = f"{panel_nombre}_{meta_nombre}_{id_img}.jpg"
            ruta_final_archivo = os.path.join(ruta_carpeta_panel, nombre_archivo)

            # Sistema anti-cortes: Si ya la descargaste antes, se la salta automáticamente
            if os.path.exists(ruta_final_archivo):
                continue

            try:
                headers = {'User-Agent': 'Mozilla/5.0'}
                res = requests.get(url_completa, headers=headers, timeout=12)

                if res.status_code == 200:
                    with open(ruta_final_archivo, 'wb') as f:
                        f.write(res.content)
                    print(f"   -> Guardado: {nombre_archivo}")
                    time.sleep(0.3)  # Pausa de seguridad para el servidor
            except Exception as e:
                print(f"    Error descargando ID {id_img}: {e}")

    print("\n---¡Dataset completo descargado y clasificado en Drive! ---")

Mounted at /content/drive
Archivo JSON cargado con éxito. Se procesarán 105 bloques de yacimientos.
Iniciando descarga masiva estructurada...

 Procesando: Cueva_de_la_Cocina (1 imágenes)

 Procesando: Abrigo_de_la_Pareja (3 imágenes)

 Procesando: Covacha_de_las_Cabras_(cavidad_derecha) (4 imágenes)

 Procesando: Covacha_de_las_Cabras_(cavidad_izquierda) (5 imágenes)

 Procesando: Les_Dogues (30 imágenes)

 Procesando: Cueva_de_las_Tortosillas (12 imágenes)

 Procesando: Abrigo_del_Ciervo_(Bco_de_las_Letras) (40 imágenes)

 Procesando: Abrigo_de_Cinto_Ventana (21 imágenes)

 Procesando: Abrigo_del_Sordo (12 imágenes)

 Procesando: Abrigo_contiguo_a_Paridera_de_las_Tajadas (3 imágenes)

 Procesando: Penascos_en_Fuente_de_Selva_Pascuala (2 imágenes)

 Procesando: Barranco_del_Cabrerizo (2 imágenes)

 Procesando: Paridera_de_las_Tajadas (5 imágenes)

 Procesando: Abrigo_de_la_Vacada (54 imágenes)

 Procesando: Abrigo_Barranco_Canas (11 imágenes)

 Procesando: Abrigo_de_las_Varias_Figuras